# ARAIP Project Part 2: CIFAR-10 CNN for the e-puck
**Arul MithunKumar (408AIAUM)** · 5FTC2068 Applied Robotics and AI Project · University of Hertfordshire

This notebook is the "off-robot" part of my project. Training a CNN needs a lot more compute than the e-puck (or the Python controller in Webots) has, so I train it here on a Colab GPU and then test it on:

1. the CIFAR-10 test set (accuracy, confusion matrix, error analysis)
2. three photos I took myself
3. frames saved by the e-puck's camera in Webots (`cnn_object_test.wbt`)

At the end I look at how to make the model light enough to run alongside the robot's controller (quantisation, pruning, and only running the CNN when the camera pre-filter sees something).

**Notebook outline**
1. Setup
2. Load and explore CIFAR-10
3. Starting point: the class model from Session 8 (`cnn_v3`)
4. My improved CNN (all layers shown) and what I changed
5. Train
6. Evaluate (class model vs mine, benchmarks, confusion matrix, mistakes)
7. Predictions on my own images
8. Predictions on the Webots camera images
9. Optimisation for the robot (TFLite, int8, pruning, speed)
10. Save results

## 1. Setup
Runtime → Change runtime type → **T4 GPU** before running. I also clone my GitHub repo so the notebook can read my own photos and the Webots captures straight from it.

In [ ]:
import os, time, json, random, gzip, shutil, pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# my project repo (Webots code + my photos + robot camera captures)
REPO_URL = "https://github.com/Mithun-Kumar10/ARAIP-Project.git"
REPO_DIR = "/content/repo"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
MY_IMAGES = pathlib.Path(REPO_DIR, "cnn", "my_images")
WEBOTS_IMAGES = pathlib.Path(REPO_DIR, "cnn", "webots_captures")
print("my images:", sorted(p.name for p in MY_IMAGES.glob("*")) if MY_IMAGES.exists() else "not found")
print("webots captures:", sorted(p.name for p in WEBOTS_IMAGES.glob("*.png")) if WEBOTS_IMAGES.exists() else "not found")

## 2. Load and explore CIFAR-10
CIFAR-10 has 60,000 colour images (32×32) in 10 classes: 50,000 for training and 10,000 for testing [Krizhevsky, 2009]. I keep 5,000 of the training images back as a validation set so I'm not tuning on the test set.

In [ ]:
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

def load_cifar10():
    # Same CIFAR-10 data. The Toronto server took ~19 min last time, so I try the
    # Hugging Face mirror first (parquet files with PNG bytes) and fall back to Keras.
    try:
        import pandas as pd, io
        from PIL import Image as _I
        base = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/"
        def part(name):
            df = pd.read_parquet(base + name)
            x = np.stack([np.array(_I.open(io.BytesIO(d["bytes"])).convert("RGB")) for d in df["img"]])
            return x, df["label"].to_numpy()
        xtr, ytr = part("train-00000-of-00001.parquet")
        xte, yte = part("test-00000-of-00001.parquet")
        print("Loaded CIFAR-10 from the Hugging Face mirror")
        return (xtr, ytr), (xte, yte)
    except Exception as e:
        print("Mirror failed (", e, ") - using keras.datasets")
        (a, b), (c, d) = keras.datasets.cifar10.load_data()
        return (a, b.ravel()), (c, d.ravel())

t0 = time.time()
(x_train_full, y_train_full), (x_test, y_test) = load_cifar10()
print(f"loading took {time.time()-t0:.0f} s")

# 45k train / 5k validation split
x_train, x_val = x_train_full[:45000], x_train_full[45000:]
y_train, y_val = y_train_full[:45000], y_train_full[45000:]

# scale pixels to 0-1
x_train = x_train.astype("float32") / 255.0
x_val   = x_val.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

print("train", x_train.shape, "val", x_val.shape, "test", x_test.shape)
print("images per class (train):", np.bincount(y_train))

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.4))
for c in range(10):
    idx = np.where(y_train == c)[0][:2]
    for r in range(2):
        axes[r, c].imshow(x_train[idx[r]]); axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=9)
plt.suptitle("Two examples from each CIFAR-10 class"); plt.tight_layout(); plt.show()

## 3. Starting point: the class model from Session 8 (`cnn_v3`)
Before building my own network I trained the model from our Session 8 notebook (`cnn_v3.ipynb`) exactly as it was given, so I have a fair baseline to improve on. Following the shapes from the Session 8 notes:

| Layer | Output shape | Why |
|---|---|---|
| Conv2D 32 (3×3) | 30×30×32 | the 3×3 kernel can only slide to 30 positions without padding |
| MaxPooling 2×2 | 15×15×32 | halves width and height, keeps the 32 channels |
| Conv2D 64 (3×3) | 13×13×64 | each of the 64 filters looks at all 32 input channels |
| MaxPooling 2×2 | 6×6×64 | |
| Conv2D 64 (3×3) | 4×4×64 | |
| Flatten | 1024 | 4 × 4 × 64 |
| Dense 64 (ReLU) | 64 | |
| Dense 10 | 10 | raw scores (logits), so the loss uses `from_logits=True` |

In class it got about 65% with SGD and about 70% with Adam after 10 epochs, and the notes pointed out that validation loss starts going up after epoch 6–7 (the start of overfitting). The only change I made is using my 5k validation split instead of the test set during training, so the test set stays unseen.

In [ ]:
from tensorflow.keras import models
baseline = models.Sequential(name="Session8_cnn_v3")
baseline.add(layers.Conv2D(32, (3, 3), activation="relu", input_shape=(32, 32, 3)))
baseline.add(layers.MaxPooling2D((2, 2)))
baseline.add(layers.Conv2D(64, (3, 3), activation="relu"))
baseline.add(layers.MaxPooling2D((2, 2)))
baseline.add(layers.Conv2D(64, (3, 3), activation="relu"))
baseline.add(layers.Flatten())
baseline.add(layers.Dense(64, activation="relu"))
baseline.add(layers.Dense(10))
baseline.summary()

baseline.compile(optimizer="adam",
                 loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=["accuracy"])
base_hist = baseline.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val), verbose=2)
base_loss, base_acc = baseline.evaluate(x_test, y_test, verbose=0)
bh = base_hist.history
best_ep = int(np.argmin(bh["val_loss"])) + 1
print(f"\nClass model test accuracy: {base_acc*100:.2f}%   params: {baseline.count_params():,}")
print(f"Final train acc {bh['accuracy'][-1]*100:.1f}% vs val acc {bh['val_accuracy'][-1]*100:.1f}% "
      f"-> gap {100*(bh['accuracy'][-1]-bh['val_accuracy'][-1]):.1f} points")
print(f"Validation loss was lowest at epoch {best_ep} and went up after that (overfitting).")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(bh["accuracy"], label="train"); ax[0].plot(bh["val_accuracy"], label="validation")
ax[0].set_title("Class model (cnn_v3): accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(bh["loss"], label="train"); ax[1].plot(bh["val_loss"], label="validation")
ax[1].axvline(best_ep - 1, ls="--", c="grey"); ax[1].text(best_ep - 1, max(bh["val_loss"]), " lowest val loss", fontsize=8)
ax[1].set_title("Class model (cnn_v3): loss"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("baseline_curves.png", dpi=120); plt.show()

### 3.1 What I changed, and why
Each change targets something I saw in the class model:

| Problem in the class model | My change |
|---|---|
| Overfits after ~6 epochs (train acc keeps rising, val loss goes up) | **Dropout** after every block (0.2 → 0.4) and **data augmentation** (random flips and shifts), so the network can't just memorise the 45k images |
| Training is slow to settle and sensitive to the learning rate | **Batch normalisation** after every convolution [Ioffe & Szegedy, 2015] |
| Only 3 conv layers, so it can't learn very detailed features | **Two 3×3 convs per block** (VGG idea), 32 → 64 → 128 filters, `padding="same"` so the image doesn't shrink before pooling |
| Flatten + Dense means many weights in the dense layer | **Global average pooling** before the dense layer, which keeps the model small for the robot |
| Fixed 10 epochs, no matter what | **ReduceLROnPlateau + EarlyStopping**, so it trains until validation loss stops improving and keeps the best weights |
| Outputs logits | **Softmax** output, so the prediction comes out as a probability I can print on the robot ("dog, 87%") |

## 4. My improved CNN
My design is a small VGG-style network [Simonyan & Zisserman, 2015]: three blocks of two 3×3 convolutions, each followed by batch normalisation [Ioffe & Szegedy, 2015] and ReLU, then max-pooling and dropout [Srivastava et al., 2014]. The number of filters doubles each block (32 → 64 → 128) while the image shrinks (32 → 16 → 8 → 4). I used global average pooling instead of a big Flatten + Dense layer because it cuts the parameter count a lot, which matters later when I try to make the model small enough for a robot.

Data augmentation (random flips, shifts) is done in the input pipeline rather than inside the model, so the saved model only has the layers needed at inference time.

In [ ]:
def conv_block(x, filters, drop, name):
    for i in (1, 2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv{i}")(x)
        x = layers.BatchNormalization(name=f"{name}_bn{i}")(x)
        x = layers.Activation("relu", name=f"{name}_relu{i}")(x)
    x = layers.MaxPooling2D(2, name=f"{name}_pool")(x)
    x = layers.Dropout(drop, name=f"{name}_drop")(x)
    return x

def build_cnn():
    inp = keras.Input(shape=(32, 32, 3), name="image")
    x = conv_block(inp, 32, 0.2, "block1")
    x = conv_block(x, 64, 0.3, "block2")
    x = conv_block(x, 128, 0.4, "block3")
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="fc1")(x)
    x = layers.Dropout(0.4, name="fc1_drop")(x)
    out = layers.Dense(10, activation="softmax", name="predictions")(x)
    return keras.Model(inp, out, name="ARAIP_CIFAR10_CNN")

model = build_cnn()
model.summary(show_trainable=True)

In [ ]:
# diagram of every layer (saved for the report)
keras.utils.plot_model(model, to_file="cnn_architecture.png", show_shapes=True,
                       show_layer_names=True, dpi=70)
from IPython.display import Image, display
display(Image("cnn_architecture.png"))

In [ ]:
# table of all layers, output shapes and parameters (for the report appendix)
rows = []
for l in model.layers:
    shape = l.output.shape if hasattr(l, "output") else ""
    rows.append((l.name, l.__class__.__name__, str(tuple(shape)), l.count_params()))
print(f"{'Layer':<16}{'Type':<22}{'Output shape':<22}{'Params':>8}")
print("-" * 68)
for r in rows: print(f"{r[0]:<16}{r[1]:<22}{r[2]:<22}{r[3]:>8,}")
print("-" * 68)
print(f"Total layers: {len(model.layers)}   Total params: {model.count_params():,}")

## 5. Train
I train with Adam [Kingma & Ba, 2015] and a batch size of 128. Two callbacks: `ReduceLROnPlateau` halves the learning rate when validation loss stops improving, and `EarlyStopping` stops training and restores the best weights if it doesn't improve for 8 epochs.

In [ ]:
BATCH = 128
EPOCHS = 60
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
], name="augment")

train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
            .shuffle(10000, seed=SEED).batch(BATCH)
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH).prefetch(AUTOTUNE)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
]

t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2)
train_minutes = (time.time() - t0) / 60
print(f"Training took {train_minutes:.1f} min for {len(history.history['loss'])} epochs")

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("training_curves.png", dpi=120); plt.show()

## 6. Evaluate
### 6.1 Accuracy on the 10,000 test images

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=256, verbose=0)
probs_test = model.predict(x_test, batch_size=256, verbose=0)
pred_test = probs_test.argmax(1)
top3 = np.mean([y in p for y, p in zip(y_test, np.argsort(-probs_test, 1)[:, :3])])
print(f"Test accuracy : {test_acc*100:.2f}%")
print(f"Top-3 accuracy: {top3*100:.2f}%")
print(f"Test loss     : {test_loss:.4f}")

### 6.2 Class model vs my model

In [ ]:
gap_base = bh["accuracy"][-1] - bh["val_accuracy"][-1]
gap_mine = h["accuracy"][-1] - h["val_accuracy"][-1]
print(f"{'':<28}{'Class cnn_v3':>14}{'My CNN':>12}")
print(f"{'Test accuracy (%)':<28}{base_acc*100:>14.2f}{test_acc*100:>12.2f}")
print(f"{'Parameters':<28}{baseline.count_params():>14,}{model.count_params():>12,}")
print(f"{'Epochs trained':<28}{len(bh['loss']):>14}{len(h['loss']):>12}")
print(f"{'Train - val accuracy gap':<28}{gap_base*100:>13.1f}%{gap_mine*100:>11.1f}%")
print(f"\nImprovement: +{(test_acc-base_acc)*100:.1f} percentage points, "
      f"error rate down from {(1-base_acc)*100:.1f}% to {(1-test_acc)*100:.1f}%")

plt.figure(figsize=(8, 3.8))
plt.plot(np.arange(1, len(bh["val_accuracy"]) + 1), bh["val_accuracy"], "o-", label="class cnn_v3 (validation)")
plt.plot(np.arange(1, len(h["val_accuracy"]) + 1), h["val_accuracy"], label="my CNN (validation)")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.grid(alpha=.3); plt.legend()
plt.title("Validation accuracy: class model vs my model")
plt.tight_layout(); plt.savefig("baseline_vs_mine.png", dpi=120); plt.show()

### 6.3 Compared with published results
The benchmark numbers come from the papers cited in my report. My model is much smaller than ResNet but I wanted to see how close a simple network could get.

In [ ]:
bench = [("Class model cnn_v3 (Session 8)", base_acc*100, baseline.count_params()),
         ("Krizhevsky 2009 (RBM features)", 78.9, None),
         ("Maxout network (Goodfellow et al. 2013)", 90.6, None),
         ("ResNet-20 (He et al. 2016)", 91.25, 0.27e6),
         ("ResNet-110 (He et al. 2016)", 93.57, 1.7e6),
         ("My CNN (this notebook)", test_acc*100, model.count_params())]
print(f"{'Model':<42}{'Test acc %':>11}{'Params':>12}")
for n, a, p in bench:
    print(f"{n:<42}{a:>11.2f}{(f'{p/1e6:.2f}M' if p else '-'):>12}")
plt.figure(figsize=(8, 3.2))
plt.barh([b[0] for b in bench], [b[1] for b in bench],
         color=["#9aa5b1"] * (len(bench) - 1) + ["#2563eb"])
plt.xlim(60, 100); plt.xlabel("CIFAR-10 test accuracy (%)"); plt.title("My CNN vs published results")
plt.tight_layout(); plt.savefig("benchmark_comparison.png", dpi=120); plt.show()

### 6.4 Per-class results and confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, pred_test, target_names=CLASSES, digits=3))
cm = confusion_matrix(y_test, pred_test)
cm_norm = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{cm_norm[i, j]*100:.0f}", ha="center", va="center",
                color="white" if cm_norm[i, j] > .5 else "black", fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (% of each true class)")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=120); plt.show()

# the biggest confusions
off = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if i != j]
print("Most common mistakes:")
for n, t, p in sorted(off, reverse=True)[:5]: print(f"  {t:>10} predicted as {p:<10} {n} times")

### 6.5 Looking at the mistakes
The most confident wrong answers are the most useful to look at, because they show what the network has actually learned.

In [ ]:
wrong = np.where(pred_test != y_test)[0]
worst = wrong[np.argsort(-probs_test[wrong].max(1))][:12]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, i in zip(axes.ravel(), worst):
    ax.imshow(x_test[i]); ax.axis("off")
    ax.set_title(f"true: {CLASSES[y_test[i]]}\npred: {CLASSES[pred_test[i]]} ({probs_test[i].max():.0%})", fontsize=8)
plt.suptitle("Most confident mistakes on the test set"); plt.tight_layout()
plt.savefig("worst_mistakes.png", dpi=120); plt.show()

## 7. Predictions on my own images
These are three photos I took myself (in `cnn/my_images` in my repo). The CNN only takes 32×32 images, so each photo is centre-cropped to a square and shrunk to 32×32, the same as the CIFAR-10 images. I show the top-3 classes for each one. This is the same idea as `load_image()` in the Session 8 notebook (resize to 32×32, scale to 0–1), except I crop to a square first so the photo isn't squashed.

If the repo clone didn't work, run the upload cell instead and pick the files.

In [ ]:
from PIL import Image as PILImage

def prepare(path_or_img):
    img = PILImage.open(path_or_img).convert("RGB") if not isinstance(path_or_img, PILImage.Image) else path_or_img.convert("RGB")
    w, h = img.size; s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    small = img.resize((32, 32), PILImage.LANCZOS)
    return img, np.asarray(small, dtype="float32") / 255.0

def show_predictions(paths, title, fname, true_labels=None):
    paths = list(paths)
    fig, axes = plt.subplots(2, len(paths), figsize=(3.6 * len(paths), 6.6), squeeze=False)
    out = []
    for k, p in enumerate(paths):
        full, small = prepare(p)
        probs = model.predict(small[None], verbose=0)[0]
        order = np.argsort(-probs)[:3]
        axes[0, k].imshow(full); axes[0, k].axis("off")
        t = f"{pathlib.Path(p).name}" + (f"\n(actually: {true_labels[k]})" if true_labels else "")
        axes[0, k].set_title(t, fontsize=9)
        axes[1, k].barh([CLASSES[o] for o in order][::-1], probs[order][::-1], color="#2563eb")
        axes[1, k].set_xlim(0, 1); axes[1, k].set_title(f"CNN: {CLASSES[order[0]]} ({probs[order[0]]:.0%})", fontsize=10)
        out.append({"image": pathlib.Path(p).name, "top3": [(CLASSES[o], round(float(probs[o]), 3)) for o in order]})
        print(f"{pathlib.Path(p).name:<28} -> " + ", ".join(f"{CLASSES[o]} {probs[o]:.1%}" for o in order))
    plt.suptitle(title); plt.tight_layout(); plt.savefig(fname, dpi=120); plt.show()
    return out

own_paths = sorted([p for p in MY_IMAGES.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]) if MY_IMAGES.exists() else []
print(len(own_paths), "own images found")

In [ ]:
# OPTIONAL: set USE_UPLOAD = True to pick the photos by hand if they weren't in the repo
USE_UPLOAD = False
if USE_UPLOAD and not own_paths:
    from google.colab import files
    up = files.upload()
    os.makedirs("my_images", exist_ok=True)
    for name, data in up.items():
        open(f"my_images/{name}", "wb").write(data)
    own_paths = sorted(pathlib.Path("my_images").glob("*"))

In [ ]:
# the label I expected for each photo comes from its file name, e.g. "cat_01.jpg" -> cat
own_true = [p.stem.split("_")[0] for p in own_paths]
own_results = show_predictions(own_paths, "CNN predictions on my own photos", "own_image_predictions.png", own_true) if own_paths else []

## 8. Predictions on the e-puck camera (Webots)
In `cnn_object_test.wbt` I put a **dog** and a **cat** (both CIFAR-10 classes, from the Webots animals library) into the arena. The `epuck_cnn_snapshot` controller drives the e-puck towards each one and saves camera frames at three distances. For this test I raised the camera to 160×120; the real e-puck camera in my search world is only 52×39, so I also test what happens at that resolution.

In [ ]:
webots_paths = sorted(WEBOTS_IMAGES.glob("*_shot*.png")) if WEBOTS_IMAGES.exists() else []
print([p.name for p in webots_paths])
webots_true = [p.stem.split("_")[0] for p in webots_paths]
webots_results = show_predictions(webots_paths, "CNN predictions on e-puck camera frames (160x120)",
                                  "webots_predictions.png", webots_true) if webots_paths else []

In [ ]:
# same frames, but first shrunk to the e-puck's real 52x39 camera resolution
def at_native(p):
    return PILImage.open(p).convert("RGB").resize((52, 39), PILImage.BILINEAR)

rows = []
for p, t in zip(webots_paths, webots_true):
    _, hi = prepare(p)
    _, lo = prepare(at_native(p))
    ph = model.predict(hi[None], verbose=0)[0]; pl = model.predict(lo[None], verbose=0)[0]
    rows.append((p.name, t, CLASSES[ph.argmax()], ph[CLASSES.index(t)], CLASSES[pl.argmax()], pl[CLASSES.index(t)]))
print(f"{'frame':<16}{'true':<6}{'160x120 pred':<16}{'P(true)':>8}   {'52x39 pred':<14}{'P(true)':>8}")
for r in rows: print(f"{r[0]:<16}{r[1]:<6}{r[2]:<16}{r[3]:>8.1%}   {r[4]:<14}{r[5]:>8.1%}")
if rows:
    print(f"\nCorrect at 160x120: {sum(r[1]==r[2] for r in rows)}/{len(rows)}   "
          f"correct at 52x39: {sum(r[1]==r[4] for r in rows)}/{len(rows)}")

In [ ]:
# what the CNN actually "sees": the 32x32 input for each robot frame
if webots_paths:
    fig, axes = plt.subplots(1, len(webots_paths), figsize=(2.2 * len(webots_paths), 2.6))
    for ax, p in zip(np.atleast_1d(axes), webots_paths):
        ax.imshow(prepare(p)[1]); ax.axis("off"); ax.set_title(p.stem, fontsize=8)
    plt.suptitle("32x32 CNN inputs from the robot camera"); plt.tight_layout()
    plt.savefig("webots_cnn_inputs.png", dpi=120); plt.show()

### 8.1 What happens with the coloured blocks?
The blocks from my search task aren't a CIFAR-10 class, but the CNN has to pick one of its 10 classes anyway (softmax always adds up to 100%). This is a good example of why a plain classifier isn't enough for the robot, and it's one of the reasons for my future work on training with my own block images or using an object detector.

In [ ]:
block_paths = sorted(WEBOTS_IMAGES.glob("*_block_found.png")) if WEBOTS_IMAGES.exists() else []
block_results = show_predictions(block_paths, "CNN on the coloured blocks (not a CIFAR-10 class)",
                                 "webots_blocks_ood.png") if block_paths else []
for r in block_results:
    print(f"{r['image']:<24} top class {r['top3'][0][0]} with {r['top3'][0][1]:.0%} confidence")

### 8.2 Does cropping to the object help?
All six robot frames above were wrong, and the wrong answers were ship, airplane and bird. My guess is that the Webots background (sky, sea and mountains) takes up most of the frame, and those are exactly the backgrounds of the ship, airplane and bird images in CIFAR-10. To test this, I crop each frame to a square around the animal before resizing it. I find the animal by masking out the sky (bright, blue-ish, top of the image) and the floor (bottom band), which is the kind of cheap region-of-interest step a robot could do with OpenCV before calling the CNN.

In [ ]:
def roi_crop(path):
    img = np.asarray(PILImage.open(path).convert("RGB")).astype(np.int16)
    h, w, _ = img.shape
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    sky = (b > r + 15) & (b > 90)                                  # blue-ish sky and sea
    floor = np.zeros_like(sky); floor[int(h * 0.80):, :] = True    # floor band at the bottom
    obj = ~(sky | floor)
    ys, xs = np.where(obj[: int(h * 0.80)])
    if len(xs) < 20:
        return PILImage.open(path).convert("RGB")
    x0, x1 = np.percentile(xs, [2, 98]).astype(int); y0, y1 = np.percentile(ys, [2, 98]).astype(int)
    cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
    side = int(max(x1 - x0, y1 - y0) * 1.1)
    side = max(16, min(side, w, h))
    xa = int(np.clip(cx - side // 2, 0, w - side)); ya = int(np.clip(cy - side // 2, 0, h - side))
    return PILImage.open(path).convert("RGB").crop((xa, ya, xa + side, ya + side))

roi_rows = []
fig, axes = plt.subplots(1, max(1, len(webots_paths)), figsize=(2.6 * max(1, len(webots_paths)), 3.2), squeeze=False)
for k, (p, t) in enumerate(zip(webots_paths, webots_true)):
    crop = roi_crop(p)
    _, x32 = prepare(crop)
    pr = model.predict(x32[None], verbose=0)[0]
    roi_rows.append({"image": p.name, "true": t, "pred": CLASSES[pr.argmax()], "conf": float(pr.max()), "p_true": float(pr[CLASSES.index(t)])})
    axes[0, k].imshow(crop); axes[0, k].axis("off")
    axes[0, k].set_title(f"{p.stem}\n{CLASSES[pr.argmax()]} ({pr.max():.0%})", fontsize=8,
                         color="green" if CLASSES[pr.argmax()] == t else "red")
plt.suptitle("Robot frames cropped to the animal (region of interest)"); plt.tight_layout()
plt.savefig("webots_roi_predictions.png", dpi=120); plt.show()
for r in roi_rows:
    print(f"{r['image']:<16}{r['true']:<6} -> {r['pred']:<11} {r['conf']:>6.1%}   P(true) {r['p_true']:.1%}")
roi_correct = sum(r["pred"] == r["true"] for r in roi_rows)
native_correct = sum(r[1] == r[4] for r in rows) if rows else None
full_correct = sum(r[1] == r[2] for r in rows) if rows else None
print(f"\nCorrect: full frame {full_correct}/{len(rows)}, at 52x39 {native_correct}/{len(rows)}, cropped to the animal {roi_correct}/{len(roi_rows)}")

### 8.3 Second Webots test: plain background, same classes as my photos
To test the background explanation properly I made a second world, `cnn_object_test_v2.wbt`. It has a plain light background and higher walls (no sky, sea or mountains), and realistic Sketchfab models of the same three classes as my own photos: a blue jay (bird), a tabby cat (cat) and a BMW E30 (automobile), all CC BY 4.0 (credits in `webots_project/models/CREDITS.md`). The e-puck turns to each one, centres it in the image, and takes a photo at three distances.

In [ ]:
WEBOTS_V2 = pathlib.Path(REPO_DIR, "cnn", "webots_captures_v2")
v2_paths = sorted(WEBOTS_V2.glob("*_shot*.png")) if WEBOTS_V2.exists() else []
v2_true = [p.stem.split("_")[0] for p in v2_paths]
v2_results = show_predictions(v2_paths, "CNN predictions: plain background, realistic models (160x120)",
                              "webots_v2_predictions.png", v2_true) if v2_paths else []
v2_rows = []
for p, t in zip(v2_paths, v2_true):
    _, hi = prepare(p); _, lo = prepare(at_native(p))
    ph = model.predict(hi[None], verbose=0)[0]; pl = model.predict(lo[None], verbose=0)[0]
    v2_rows.append((p.name, t, CLASSES[ph.argmax()], ph[CLASSES.index(t)], CLASSES[pl.argmax()], pl[CLASSES.index(t)]))
print(f"{'frame':<24}{'true':<12}{'160x120 pred':<14}{'P(true)':>8}   {'52x39 pred':<12}{'P(true)':>8}")
for r in v2_rows: print(f"{r[0]:<24}{r[1]:<12}{r[2]:<14}{r[3]:>8.1%}   {r[4]:<12}{r[5]:>8.1%}")
v2_correct = sum(r[1] == r[2] for r in v2_rows); v2_native = sum(r[1] == r[4] for r in v2_rows)
print(f"\nPlain background: {v2_correct}/{len(v2_rows)} correct at 160x120, {v2_native}/{len(v2_rows)} at 52x39 "
      f"(first world with sky/sea background: {full_correct}/{len(rows)})")

### 8.4 Save the results so far
This downloads the figures and numbers up to here, so nothing is lost if the runtime disconnects during the (slow) optimisation section.

In [ ]:
gap_mine = h["accuracy"][-1] - h["val_accuracy"][-1]
off = sorted(((int(cm[i, j]), CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if i != j), reverse=True)
partial = {
    "test_accuracy": round(float(test_acc), 4), "top3_accuracy": round(float(top3), 4),
    "params": int(model.count_params()), "epochs": len(h["loss"]), "train_minutes": round(train_minutes, 1),
    "train_val_gap_mine": round(float(gap_mine), 4),
    "baseline_cnn_v3": {"test_accuracy": round(float(base_acc), 4), "params": int(baseline.count_params()),
                        "train_val_gap": round(float(bh["accuracy"][-1] - bh["val_accuracy"][-1]), 4), "best_val_epoch": best_ep},
    "top_mistakes": [[t, p, n] for n, t, p in off[:5]],
    "own_images": own_results, "webots_images": webots_results, "webots_blocks": block_results,
    "native_correct": native_correct, "roi": roi_rows, "roi_correct": roi_correct,
    "webots_v2": v2_results, "webots_v2_correct": v2_correct, "webots_v2_native_correct": v2_native,
}
json.dump(partial, open("results_summary.json", "w"), indent=2)
model.save("cifar10_cnn.keras")
!zip -q araip_cnn_outputs_part1.zip *.png results_summary.json
from google.colab import files
files.download("araip_cnn_outputs_part1.zip")
print(json.dumps({k: v for k, v in partial.items() if k not in ("own_images", "webots_images", "webots_blocks", "roi")}, indent=2))

## 9. Optimisation for running on the robot
The controller in Webots runs every 32 ms, so if the CNN takes longer than that the robot's control loop slows down and it can't react to walls in time. I tested four ways of making the model cheaper:

| Method | Idea |
|---|---|
| TFLite float32 | same weights, but a lighter runtime made for edge devices |
| Dynamic-range quantisation | weights stored as 8-bit ints, about 4× smaller [Jacob et al., 2018] |
| Full int8 quantisation | weights **and** activations in int8, needs a small calibration set |
| Magnitude pruning (50% / 75%) + fine-tune | zero the smallest weights, then retrain a bit so accuracy recovers [Han et al., 2016] |

For each one I measure test accuracy, file size and single-image latency on the Colab CPU (the robot won't have a GPU).

In [ ]:
os.makedirs("export", exist_ok=True)

def to_tflite(keras_model, mode):
    """Convert straight from the Keras model through a concrete function with a fixed
    1x32x32x3 input (one camera frame at a time, like on the robot). My first run used
    model.export() -> SavedModel, which crashed with an InvalidArgumentError."""
    @tf.function(input_signature=[tf.TensorSpec([1, 32, 32, 3], tf.float32)])
    def serve(x):
        return keras_model(x, training=False)
    conv = tf.lite.TFLiteConverter.from_concrete_functions([serve.get_concrete_function()], keras_model)
    if mode in ("dynamic", "int8"):
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    if mode == "int8":
        def rep():
            for i in range(300):
                yield [x_train[i:i+1]]
        conv.representative_dataset = rep
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.uint8
        conv.inference_output_type = tf.uint8
    return conv.convert()

def gz_size_kb(b):
    return len(gzip.compress(b)) / 1024

def tflite_eval(tfl_bytes, n=10000):
    it = tf.lite.Interpreter(model_content=tfl_bytes); it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    correct, times = 0, []
    for i in range(n):
        x = x_test[i:i+1]
        if inp["dtype"] == np.uint8:
            s, z = inp["quantization"]; x = np.clip(np.round(x / s + z), 0, 255).astype(np.uint8)
        t0 = time.perf_counter(); it.set_tensor(inp["index"], x); it.invoke()
        y = it.get_tensor(out["index"])[0]; times.append(time.perf_counter() - t0)
        correct += int(np.argmax(y) == y_test[i])
    return correct / n, 1000 * np.median(times)

def keras_latency(m, n=200):
    with tf.device("/CPU:0"):
        m.predict(x_test[:1], verbose=0)
        ts = []
        for i in range(n):
            t0 = time.perf_counter(); m(x_test[i:i+1], training=False); ts.append(time.perf_counter() - t0)
    return 1000 * np.median(ts)

results = []
model.save("export/cifar10_cnn.keras")
kb = open("export/cifar10_cnn.keras", "rb").read()
results.append(("Keras float32 (baseline)", test_acc, len(kb)/1024, gz_size_kb(kb), keras_latency(model)))

for mode, name in [("float", "TFLite float32"), ("dynamic", "TFLite dynamic-range int8"), ("int8", "TFLite full int8")]:
    try:
        b = to_tflite(model, mode)
        open(f"export/cifar10_cnn_{mode}.tflite", "wb").write(b)
        acc, ms = tflite_eval(b)
        results.append((name, acc, len(b)/1024, gz_size_kb(b), ms))
        print(f"{name:<28} acc {acc*100:.2f}%  size {len(b)/1024:.0f} KB  latency {ms:.2f} ms")
    except Exception as e:          # one failed variant shouldn't stop the others
        print(f"{name}: conversion failed ({type(e).__name__}: {str(e)[:200]})")

In [ ]:
# magnitude pruning: zero the smallest |w| in every conv/dense kernel, then fine-tune with the mask held
def prune_and_finetune(sparsity, epochs=2):
    m = keras.models.clone_model(model); m.set_weights(model.get_weights())
    masks = {}
    for l in m.layers:
        if isinstance(l, (layers.Conv2D, layers.Dense)):
            w = l.kernel.numpy(); thr = np.quantile(np.abs(w), sparsity)
            masks[l.name] = (np.abs(w) > thr).astype("float32")
            l.kernel.assign(w * masks[l.name])

    class KeepMask(keras.callbacks.Callback):
        def on_train_batch_end(self, batch, logs=None):
            for l in m.layers:
                if l.name in masks: l.kernel.assign(l.kernel * masks[l.name])

    acc_before = m.evaluate(x_test, y_test, verbose=0, batch_size=256)[1]
    m.compile(optimizer=keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[KeepMask()], verbose=0)
    acc_after = m.evaluate(x_test, y_test, verbose=0, batch_size=256)[1]
    zeros = np.mean([np.mean(l.kernel.numpy() == 0) for l in m.layers if l.name in masks])
    return m, acc_before, acc_after, zeros

for sp in (0.5, 0.75):
    pm, a0, a1, z = prune_and_finetune(sp)
    try:
        b = to_tflite(pm, "dynamic")
    except Exception as e:
        print(f"pruned {sp}: conversion failed ({type(e).__name__}: {str(e)[:200]})"); continue
    acc, ms = tflite_eval(b)
    name = f"Pruned {int(sp*100)}% + dynamic int8"
    results.append((name, acc, len(b)/1024, gz_size_kb(b), ms))
    print(f"{name:<28} before fine-tune {a0*100:.2f}% -> after {a1*100:.2f}% | "
          f"TFLite acc {acc*100:.2f}%  gzip {gz_size_kb(b):.0f} KB  ({z*100:.0f}% zeros)")

In [ ]:
print(f"{'Model':<30}{'Accuracy %':>11}{'Size KB':>10}{'gzip KB':>10}{'CPU ms':>9}{'Max FPS':>9}")
print("-" * 79)
for n, a, s, g, ms in results:
    print(f"{n:<30}{a*100:>11.2f}{s:>10.0f}{g:>10.0f}{ms:>9.2f}{1000/ms:>9.0f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
names = [r[0].replace(" + ", "\n+ ").replace("TFLite ", "TFLite\n") for r in results]
ax[0].bar(names, [r[1]*100 for r in results], color="#2563eb"); ax[0].set_ylim(min(r[1] for r in results)*100 - 3, 100)
ax[0].set_title("Test accuracy (%)")
ax[1].bar(names, [r[3] for r in results], color="#16a34a"); ax[1].set_title("Compressed size (KB, gzip)")
ax[2].bar(names, [r[4] for r in results], color="#ea580c"); ax[2].set_title("CPU latency per image (ms)")
ax[2].axhline(32, ls="--", c="k", lw=1); ax[2].text(0, 33, "32 ms control step", fontsize=8)
for a in ax: a.tick_params(axis="x", labelsize=7, rotation=0)
plt.tight_layout(); plt.savefig("optimisation_tradeoffs.png", dpi=120); plt.show()

### 9.1 Event-triggered inference (the camera pre-filter)
The other half of my optimisation proposal isn't about the model at all, it's about *when* it runs. My Webots controller already has a cheap pre-filter that samples every 4th pixel and only does the full colour scan if it spots something. I also moved the colour detection to OpenCV (`cvtColor` + `inRange`, as in Session 9), which made each frame about 10× cheaper than my pure Python loop. The batch test numbers below come from `results/runs.csv` in my Webots project (10 random layouts per controller). If the CNN is only called on frames that pass the pre-filter, the average cost per control step drops by the same fraction.

In [ ]:
import pandas as pd
csv_path = pathlib.Path(REPO_DIR, "webots_project", "results", "runs.csv")
if csv_path.exists():
    runs = pd.read_csv(csv_path)
    display(runs.groupby("label").agg(success=("outcome", lambda o: (o == "success").sum()), mean_time_s=("time_s", "mean"), worst_time_s=("time_s", "max"), vision_ms=("vision_ms", "mean")).round(2))
    v2 = runs[runs.label == "v2_opencv"]
    gate = float((v2.full_scans / v2.frames).mean()) if len(v2) else 1.0
else:
    gate = 0.35; print("runs.csv not found, using an example value")
best_ms = min(r[4] for r in results)
print(f"Fraction of frames that pass the pre-filter: {gate:.0%}")
print(f"Fastest model latency: {best_ms:.2f} ms -> average CNN cost per 32 ms step "
      f"if run every frame: {best_ms:.2f} ms, with pre-filter gating: {best_ms*gate:.2f} ms")

## 10. Save everything
These files go into my report and back into the Webots project (`cnn/cifar10_cnn.tflite`).

In [ ]:
summary = dict(partial)
summary.update({
    "test_accuracy": round(float(test_acc), 4), "top3_accuracy": round(float(top3), 4),
    "baseline_cnn_v3": {"test_accuracy": round(float(base_acc), 4), "params": int(baseline.count_params()),
                        "train_val_gap": round(float(gap_base), 4), "best_val_epoch": best_ep},
    "params": int(model.count_params()), "epochs": len(history.history["loss"]),
    "train_minutes": round(train_minutes, 1),
    "own_images": own_results, "webots_images": webots_results, "webots_blocks": block_results,
    "optimisation": [{"model": n, "acc": round(float(a), 4), "size_kb": round(s, 1),
                      "gzip_kb": round(g, 1), "cpu_ms": round(float(ms), 3)} for n, a, s, g, ms in results],
    "prefilter_gate": round(float(gate), 3),
})
json.dump(summary, open("results_summary.json", "w"), indent=2)
if os.path.exists("export/cifar10_cnn_dynamic.tflite"):
    shutil.copy("export/cifar10_cnn_dynamic.tflite", "cifar10_cnn.tflite")
!zip -q -r araip_cnn_outputs.zip *.png results_summary.json cifar10_cnn.tflite export/cifar10_cnn.keras 2>/dev/null
print(json.dumps({k: v for k, v in summary.items() if k not in ("own_images", "webots_images")}, indent=2))
from google.colab import files
files.download("araip_cnn_outputs.zip")